In [1]:
import os
import gc
import random
import timm
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, log_loss
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from scipy.stats import randint, uniform

# Settings
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

IMG_SIZE = 224
BATCH_SIZE = 32
N = 10   # ways
K = 3   # shots
Q = 4   # queries per class
EPISODES = 1000  # Equivalent to few epochs

c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Dataset 
class ProtoDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = np.array(file_paths)
        self.labels = np.array(labels)
        self.transform = transform
        self.classes = np.unique(labels)
        self.class_to_indices = {c: np.where(labels == c)[0] for c in self.classes}

    def sample_episode(self, N, K, Q):
        """Create one training episode."""
        selected_classes = np.random.choice(self.classes, N, replace=False)

        support_images = []
        support_labels = []
        query_images = []
        query_labels = []

        for i, cls in enumerate(selected_classes):
            idxs = np.random.choice(self.class_to_indices[cls], K + Q, replace=False)

            support_idxs = idxs[:K]
            query_idxs = idxs[K:]

            for idx in support_idxs:
                img = Image.open(self.file_paths[idx]).convert("RGB")
                if self.transform: img = self.transform(img)
                support_images.append(img)
                support_labels.append(i)

            for idx in query_idxs:
                img = Image.open(self.file_paths[idx]).convert("RGB")
                if self.transform: img = self.transform(img)
                query_images.append(img)
                query_labels.append(i)

        return (
            torch.stack(support_images),
            torch.tensor(support_labels),
            torch.stack(query_images),
            torch.tensor(query_labels)
        )

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [3]:
# Load Dataset
# Define DATA_PATH and CLASSES
DATA_PATH = "C:/Users/MSC055/Desktop/Sabrina/Eye Image Dataset"
CLASSES = sorted(os.listdir(DATA_PATH))

file_paths, labels = [], []
for idx, cls in enumerate(CLASSES):
    cls_path = os.path.join(DATA_PATH, cls)
    for f in os.listdir(cls_path):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            file_paths.append(os.path.join(cls_path, f))
            labels.append(idx)

file_paths = np.array(file_paths)
labels = np.array(labels)
# Train/Val/Test split
X_train, X_temp, y_train, y_temp = train_test_split(file_paths, labels, test_size=0.3, stratify=labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

print(f"Train_size: {len(X_train)}\nVal_size: {len(X_val)}\nTest_size: {len(X_test)}")



Train_size: 3734
Val_size: 800
Test_size: 801


In [4]:
# Prototypical Network
class ProtoNetwork(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        f = self.backbone(x)
        if len(f.shape) > 2:
            f = torch.flatten(f, 1)
        return f

class ProtoLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self._printed = False

    def forward(self, support, support_labels, query, query_labels, model):
        # Get embeddings
        support_emb = model(support)   # [N*K, D]
        query_emb = model(query)       # [N*Q, D]
        if not self._printed:
            print("Support embedding shape:", support_emb.shape)
            print("Query embedding shape:", query_emb.shape)
            self._printed = True

        # Compute class prototypes
        prototypes = []
        for cls in torch.unique(support_labels):
            cls_emb = support_emb[support_labels == cls]
            prototypes.append(cls_emb.mean(dim=0))
        prototypes = torch.stack(prototypes)  # [N, D]

        # Compute distances
        dists = torch.cdist(query_emb, prototypes)   # [N*Q, N]

        # CE loss (lower distance = higher score)
        logits = -dists
        return self.ce(logits, query_labels)

def extract_embeddings(model, file_paths, transform):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for path in tqdm(file_paths, desc="Extracting Embeddings"):
            img = Image.open(path).convert("RGB")
            img = transform(img).unsqueeze(0).to(DEVICE)
            feat = model(img)  # use proto_model directly
            embeddings.append(feat.cpu().numpy())
    return np.vstack(embeddings)


In [ ]:
# Backbone Models
backbones = {
    #"convit_small": lambda pretrained=True: timm.create_model("convit_small", pretrained=pretrained),
    "efficientvit_b0": lambda pretrained=True: timm.create_model("efficientvit_b0", pretrained=pretrained)
}

# Symbolic Models
symbolic_models = {
    "RandomForest": (
        RandomForestClassifier(random_state=SEED),
        {
            "n_estimators": randint(50, 100),
            "max_depth": randint(5, 50),
            "min_samples_split": randint(2, 20),
            "min_samples_leaf": randint(1, 10),
            "max_features": ["sqrt", "log2", None],
            "bootstrap": [True, False]
        }
    ),
    "XGBoost": (
        xgb.XGBClassifier(eval_metric="mlogloss", random_state=SEED),
        {
            "n_estimators": randint(50, 100),
            "max_depth": randint(3, 15),
            "learning_rate": uniform(0.01, 0.3),
            "subsample": uniform(0.5, 0.5),
            "colsample_bytree": uniform(0.5, 0.5),
            "gamma": uniform(0, 5),
            "reg_alpha": uniform(0, 1),
            "reg_lambda": uniform(0, 1)
        }
    ),
    
    "LightGBM": (
        lgb.LGBMClassifier(verbose=-1, random_state=SEED),
        {
            "n_estimators": randint(50, 100),
            "max_depth": randint(3, 15),
            "learning_rate": uniform(0.01, 0.3),
            "subsample": uniform(0.5, 0.5),
            "colsample_bytree": uniform(0.5, 0.5),
            "gamma": uniform(0, 5),
            "reg_alpha": uniform(0, 1),
            "reg_lambda": uniform(0, 1)
        }
    ),
    "CatBoost": (
        cb.CatBoostClassifier(verbose=0,random_state=SEED),
        {
            "iterations": randint(50, 100),
            "depth": randint(3, 12),
            "learning_rate": uniform(0.01, 0.3),
            "l2_leaf_reg": uniform(1, 10),
            "border_count": randint(32, 255),
            "random_strength": uniform(0, 10)
        }
    ),
    "AdaBoost": (
        AdaBoostClassifier(random_state=SEED),
        {
           "n_estimators": randint(50, 100),
            "learning_rate": uniform(0.01, 1.0)
        }
    )
}


In [ ]:
# Training Loop
results_summary = []

for backbone_name, backbone_func in backbones.items():
    print(f"\nBackbone: {backbone_name}")
    
    backbone_model = backbone_func(pretrained=True)
    # Remove classification layer (depending on the architecture)
    if hasattr(backbone_model, "fc"):
        backbone_model.fc = nn.Identity()
    elif hasattr(backbone_model, "classifier"):
        backbone_model.classifier = nn.Identity()
    elif hasattr(backbone_model, "head"):
        backbone_model.head = nn.Identity()
    elif hasattr(backbone_model, "heads"):
        backbone_model.heads = nn.Identity()
    elif hasattr(backbone_model, "classifier_head"):
        backbone_model.classifier_head = nn.Identity()
        
    backbone_model.to(DEVICE).eval()
    
    proto_model = ProtoNetwork(backbone_model).to(DEVICE)
    proto_loss = ProtoLoss()
    optimizer = torch.optim.Adam(proto_model.parameters(), lr=1e-4)

   

    proto_dataset = ProtoDataset(X_train, y_train, transform=train_transform)

    print("Training Prototypical Network...")

    for episode in range(EPISODES):
        support_x, support_y, query_x, query_y = proto_dataset.sample_episode(N, K, Q)

        support_x = support_x.to(DEVICE)
        query_x   = query_x.to(DEVICE)
        support_y = support_y.to(DEVICE)
        query_y   = query_y.to(DEVICE)

        loss = proto_loss(support_x, support_y, query_x, query_y, proto_model)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if episode % 50 == 0:
            print(f"[ProtoNet] Episode {episode}/{EPISODES} | Prototypical Loss: {loss.item():.4f}")

     
    # Extract features
    X_train_feat = extract_embeddings(proto_model, X_train, val_transform)
    X_val_feat = extract_embeddings(proto_model, X_val, val_transform)
    original_dim = X_train_feat.shape[1]
    print(f"Original feature dimension: {original_dim}")

    for sym_name, (clf, param_dist) in symbolic_models.items():
        print(f"\nSymbolic Model: {sym_name}")
        search = RandomizedSearchCV(clf, param_distributions=param_dist, n_iter=10, cv=3,
                                    scoring="accuracy", random_state=SEED, n_jobs=1)
        search.fit(X_train_feat, y_train)
        tuned_model = search.best_estimator_
        best_params = search.best_params_
        
        y_train_pred = tuned_model.predict(X_train_feat)
        y_val_pred = tuned_model.predict(X_val_feat)
        
        y_train_proba = tuned_model.predict_proba(X_train_feat)
        y_val_proba = tuned_model.predict_proba(X_val_feat)
        
        train_loss_calc = log_loss(y_train, y_train_proba)
        val_loss_calc = log_loss(y_val, y_val_proba)
        
        train_acc = accuracy_score(y_train, y_train_pred)
        val_acc = accuracy_score(y_val, y_val_pred)
        overall_acc = val_acc
        precision = precision_score(y_val, y_val_pred, average='macro', zero_division=0)
        recall = recall_score(y_val, y_val_pred, average='macro', zero_division=0)
        f1 = f1_score(y_val, y_val_pred, average='macro', zero_division=0)
        class_report = classification_report(y_val, y_val_pred, zero_division=0)
        conf_matrix = confusion_matrix(y_val, y_val_pred)

        print(f"\n{backbone_name} + {sym_name}")
        print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Train Loss: {train_loss_calc:.4f} | Val Loss: {val_loss_calc:.4f}")
        print(f"Accuracy: {overall_acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1-score: {f1:.4f}")
        print(f"Best Params: {best_params}")
        print("Classification Report:\n", class_report)
        print("Confusion Matrix:\n", conf_matrix)
        
        results_summary.append({
            "backbone": backbone_name,
            "symbolic_model": sym_name,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "train_loss": train_loss_calc,
            "val_loss": val_loss_calc,
            "overall_acc": overall_acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "conf_matrix": conf_matrix,
            "class_report": class_report,
            "best_params": best_params,
            "model": tuned_model,
            "backbone_model": proto_model
        })

    # Summary 
    print(f"\nCompleted backbone {backbone_name} summary:")
    for r in [res for res in results_summary if res["backbone"] == backbone_name]:
        print(f"{r['symbolic_model']:<15} | Train Acc : {r['train_acc']:.4f} | "
              f"Train Loss : {r['train_loss']:.4f} | Val Acc: {r['val_acc']:.4f} | "
              f"Val Loss: {r['val_loss']:.4f} | Accuracy : {r['overall_acc']:.4f}| "
              f"Precision : {r['precision']:.4f} | Recall : {r['recall']:.4f} | F1: {r['f1']:.4f}")



Backbone: efficientvit_b0
Training Prototypical Network...
Support embedding shape: torch.Size([30, 6272])
Query embedding shape: torch.Size([40, 6272])
[ProtoNet] Episode 0/1000 | Prototypical Loss: 7.6235
[ProtoNet] Episode 50/1000 | Prototypical Loss: 1.7215
[ProtoNet] Episode 100/1000 | Prototypical Loss: 1.6328
[ProtoNet] Episode 150/1000 | Prototypical Loss: 1.9116
[ProtoNet] Episode 200/1000 | Prototypical Loss: 1.8438
[ProtoNet] Episode 250/1000 | Prototypical Loss: 1.7067
[ProtoNet] Episode 300/1000 | Prototypical Loss: 1.9624
[ProtoNet] Episode 350/1000 | Prototypical Loss: 1.5465
[ProtoNet] Episode 400/1000 | Prototypical Loss: 1.8770
[ProtoNet] Episode 450/1000 | Prototypical Loss: 1.7996
[ProtoNet] Episode 500/1000 | Prototypical Loss: 1.6773
[ProtoNet] Episode 550/1000 | Prototypical Loss: 1.4116
[ProtoNet] Episode 600/1000 | Prototypical Loss: 1.2609
[ProtoNet] Episode 650/1000 | Prototypical Loss: 1.4441
[ProtoNet] Episode 700/1000 | Prototypical Loss: 1.5659
[ProtoNet

Extracting Embeddings: 100%|██████████| 800/800 [00:20<00:00, 39.05it/s]


Original feature dimension: 6272

Symbolic Model: RandomForest

efficientvit_b0 + RandomForest
Train Acc: 0.7019 | Val Acc: 0.6050 | Train Loss: 0.8727 | Val Loss: 1.0789
Accuracy: 0.6050 | Precision: 0.5130 | Recall: 0.4555 | F1-score: 0.4656
Best Params: {'bootstrap': True, 'max_depth': 6, 'max_features': None, 'min_samples_leaf': 7, 'min_samples_split': 13, 'n_estimators': 57}
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        15
           1       0.77      0.94      0.85       226
           2       0.00      0.00      0.00        19
           3       0.47      0.63      0.54       202
           4       0.50      0.52      0.51       153
           5       0.00      0.00      0.00        67
           6       0.61      0.57      0.59        75
           7       1.00      1.00      1.00         3
           8       0.86      0.32      0.46        19
           9       0.92      0.57      0.71        2

c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with featur


efficientvit_b0 + LightGBM
Train Acc: 0.9344 | Val Acc: 0.5938 | Train Loss: 0.0972 | Val Loss: 1.6604
Accuracy: 0.5938 | Precision: 0.6305 | Recall: 0.4602 | F1-score: 0.4946
Best Params: {'colsample_bytree': np.float64(0.6943386448447411), 'gamma': np.float64(1.3567451588694794), 'learning_rate': np.float64(0.2586212527455788), 'max_depth': 12, 'n_estimators': 74, 'reg_alpha': np.float64(0.28093450968738076), 'reg_lambda': np.float64(0.5426960831582485), 'subsample': np.float64(0.5704621124873813)}
Classification Report:
               precision    recall  f1-score   support

           0       0.50      0.07      0.12        15
           1       0.81      0.91      0.85       226
           2       0.75      0.16      0.26        19
           3       0.48      0.57      0.52       202
           4       0.46      0.49      0.47       153
           5       0.32      0.10      0.16        67
           6       0.56      0.60      0.58        75
           7       1.00      0.67   

'     \n    # Clear memory before moving to the next backbone\n    del backbone_model, proto_model, X_train_feat, X_val_feat\n    torch.cuda.empty_cache()\n    gc.collect()\n    print(f"Memory cleared successfully after backbone: {backbone_name}")\n\n\nbest_combo = max(results_summary, key=lambda x: (x["val_acc"], -x["val_loss"]))\nprint(f"\nBest Backbone–Symbolic Pair:")\nprint(f"Backbone: {best_combo[\'backbone\']}")\nprint(f"Symbolic Model: {best_combo[\'symbolic_model\']}")\nprint(f"Training Accuracy: {best_combo[\'train_acc\']:.4f}, Training Loss: {best_combo[\'train_loss\']:.4f}")\nprint(f"Validation Accuracy: {best_combo[\'val_acc\']:.4f}, Validation Loss: {best_combo[\'val_loss\']:.4f}")\nprint(f"Best Params: {best_combo[\'best_params\']}")\nprint(f"Overall Accuracy: {best_combo[\'overall_acc\']:.4f}")\nprint(f"Precision: {best_combo[\'precision\']:.4f}, Recall: {best_combo[\'recall\']:.4f}, F1: {best_combo[\'f1\']:.4f}")\nprint("Classification Report:\n", best_combo["class_rep